# Exploratory Data Analysis
## image structure

In [6]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import re

### Data loading

In [27]:
data_path = Path.cwd().parent.joinpath('data/vial')
if not data_path.exists():
    raise FileExistsError("could not find the vial folder... Please check")

all_files = []
count_images = 0
#capture le dossier de split, eventuellement la catégorie si existante (bad, good etc.) le filename (ie l'id de l'image), le defect existant et l'extension de l'image
regex = r'/vial/(?P<split>[^/]+)(?:/(?P<category>([^/]+)))?/(?P<filename>[\d]+)\_(?P<defect>[^.]+)\.(?P<extension>[^$]+)'
regex_mask = r'/vial/(?P<split>[^/]+)(?:/(?P<type>([^/]+)))?/(?P<category>[^/]+)/(?P<filename>[\d]+)\_(?P<defect>[^.]+)_mask\.(?P<extension>[^$]+)'
for sub in data_path.iterdir():
    for imgs in sub.rglob("*.png"):
        count_images+=1
        match = re.search(regex, str(imgs))
        match_mask = re.search(regex_mask, str(imgs))
        if match is not None:
            all_files.append({
                'data_type' : 'image',
                'split' : match.group('split'),
                'category' : match.group('category'),
                'filename' : match.group('filename'),
                'defect' : match.group('defect'),
                'extension' : match.group('extension'),
                'path' : imgs
            })
        elif match_mask is not None:
            all_files.append({
                'data_type' : 'mask',
                'split' : match_mask.group('split'),
                'type' : match_mask.group('type'),
                'category' : match_mask.group('category'),
                'filename' : match_mask.group('filename'),
                'defect' : match_mask.group('defect'),
                'extension' : match_mask.group('extension'),
                'path' : imgs
            })
        else:
            print('imgs')
df = pd.DataFrame(all_files)

In [34]:
df["data_type"].value_counts(dropna=False)

data_type
image    1024
mask      105
Name: count, dtype: int64

In [35]:
df["split"].value_counts(dropna=False)

split
train                 291
test_private_mixed    276
test_private          276
test_public           245
validation             41
Name: count, dtype: int64

In [36]:
df["category"].value_counts(dropna=False)


category
NaN     552
good    367
bad     210
Name: count, dtype: int64

In [37]:
df[df["category"].isna()]["split"].value_counts(dropna=False)

split
test_private_mixed    276
test_private          276
Name: count, dtype: int64

First observation : here we see that test_private doesn't offer categories (reminder : good or bad).
Now let's dive a little deeper.

In [43]:
df[df['split']=="train"]['category'].unique()

<StringArray>
['good']
Length: 1, dtype: str

Here we see that the training data are only "good".

In [44]:
df[df['split']=="validation"]['category'].unique()

<StringArray>
['good']
Length: 1, dtype: str

and validation is good only as well. Let's look at test data!

In [49]:
df[df['split']=='test_public']['data_type'].value_counts()

data_type
image    140
mask     105
Name: count, dtype: int64

We have, in the test_public dataset 140 images and 105 masks. Let's look into data categories now.

In [60]:
df[df['split']=='test_public'].groupby(['category','data_type'], dropna=False).size()

category  data_type
bad       image        105
          mask         105
good      image         35
dtype: int64

as anticipated, masks are associated with bad samples. Thus we have 105 defects, with the associated mask. Now let's look at the private tests datasets.

In [74]:
df[df['split'].isin(['test_private','test_private_mixed'])]['category'].unique()

<StringArray>
[nan]
Length: 1, dtype: str

In [75]:
df[df['split'].isin(['test_private','test_private_mixed'])]['data_type'].unique()

<StringArray>
['image']
Length: 1, dtype: str

test_private and test_private_mixed don't have categories, or masks. They only are images.

To summarize, at this step:
- Train and validation data are only good data
- public test data are good and bad data. Bad data handily come with a mask!
- private tests are not labeled, only images.

However, from the name of the sets, I can infer that test_private only has one type of data whereas test_private_mixed has a bit of both : good and bad.